## Super()

In [ ]:
# %load_ext autoreload
# %autoreload 2

In [ ]:
# super() used inside a method has parameters filled automatically
class A:
  _class = 'A'

class B(A):
  _class = 'B'

class C(B):
  _class = 'C'
  def inst_meth(self):
    print(super()._class)

In [ ]:
C.__mro__

In [ ]:
super(B, C)._class

In [ ]:
C().inst_meth()

## 1 init subclass

In [ ]:
class A:
  _class = 'A'
  # never runs for the class its defined in
  def __init_subclass__(cls):
    print('A.__init_subclass__ called for:', cls)

class B(A):
  _class = 'B'

class C(B):
  _class = 'C'

In [ ]:
class A:
  _class = 'A'
  # never runs for the class its defined in
  def __init_subclass__(cls):
    print('A.__init_subclass__ called for:', cls)
    # cls.parent = super(cls ,cls)._class

class B(A):
  _class = 'B'
  def __init_subclass__(cls, **kwargs):
    super().__init_subclass__(**kwargs)
    print('B.__init_subclass__ called for:', cls)

class C(B):
  _class = 'C'

## Example: config

In [ ]:
from pgt.config import CONFIG


class DataSource:
  config = CONFIG['DataSource']

  def __init_subclass__(cls):
    cls.config = {**cls.config, **CONFIG.get(cls.__name__, {})}

class LocalSource(DataSource):
  pass

class CSVSource(LocalSource):
  pass

In [ ]:
DataSource.config

In [ ]:
LocalSource.config

In [ ]:
CSVSource.config

### registry example

In [ ]:
from pathlib import Path


class DataSource:
  registry = {}
  extensions = ()

  def __init_subclass__(cls):
    for ext in cls.extensions:
      DataSource.registry[ext] = cls

  @staticmethod
  def load(path):
    path = Path(path)
    return DataSource.registry[path.suffix]().read(path)

class CSVSource(DataSource):
  extensions = ('.csv', '.txt')

  def read(self, path):
    ...

class ParquetSource(DataSource):
  extensions = ('.parquet',)

  def read(self, path):
    ...


df = DataSource.load('file.csv')

## 2 Exec

In [ ]:
exec('c=5')

In [ ]:
c

### Where is it used inside Python -dataclasses, namedtuples prior to ~3.7

## Example

In [ ]:
import copy
import csv
from dataclasses import make_dataclass
from decimal import Decimal
from pathlib import Path

from pgt.exec_config import CONFIG

TYPE_MAP = {'str': str, 'int': int, 'float': float, 'Decimal': Decimal}


class OHLCVMixin:
  __slots__ = ()

  @property
  def hl2(self):
    return (self.low + self.high) / 2


class CSVDataSource:
  def __init__(self, source_file: Path):
    self.source_file = source_file
    self.selected_columns = None

  def select(self, *columns):
    new = copy.copy(self)
    new.selected_columns = list(columns)
    return new

  def _generate_row_convertor(self, header):
    schema = CONFIG['CSVDataSource']['schema']
    columns = self.selected_columns or [col for col in header if col in schema]
    indices = [header.index(col) for col in columns]
    types = [TYPE_MAP[schema[col]] for col in columns]

    Row = make_dataclass('Row', zip(columns, types, strict=True), bases=(OHLCVMixin,), slots=True)

    converted = (f'{t.__name__}(row[{i}])' for i, t in zip(indices, types, strict=True))
    code = (
      'def row_convertor(row):\n'
      f"  return Row({', '.join(converted)})"
    )

    globs = {'Row': Row, 'Decimal': Decimal}
    locs = {}
    exec(code, globals=globs, locals=locs)
    self.row_convertor = locs['row_convertor']
    print(code)

  def __iter__(self):
    with open(self.source_file, newline='') as f:
      reader = csv.reader(f)
      header = next(reader)
      self._generate_row_convertor(header)
      for row in reader:
        yield self.row_convertor(row)


class ZipCSVDataSource(CSVDataSource):
  def __iter__(self):
    with open(self.source_file, newline='') as f:
      reader = csv.reader(f)
      header = next(reader)

      schema = CONFIG['CSVDataSource']['schema']
      columns = self.selected_columns or [col for col in header if col in schema]
      indices = [header.index(col) for col in columns]
      types = [TYPE_MAP[schema[col]] for col in columns]

      Row = make_dataclass('Row', zip(columns, types, strict=True), bases=(OHLCVMixin,), slots=True)

      for row in reader:
        yield Row(*[t(row[i]) for t, i in zip(types, indices, strict=True)])

In [ ]:
file = Path('C:\\Users\\adam.jurco\\Documents\\Git\\PythonGuild\\data\\df_BTCUSDT_1m_futures_20250101_20251231.csv')
v1 = CSVDataSource(file)
v2 = ZipCSVDataSource(file)

In [ ]:
# open_time; close
for row in v1.select('close', 'high'):
  break

In [ ]:
# uv run python -m pgt.exec_timeit
from pgt.exec_timeit import load, plot

fig = plot(load()) # relod results from ./results/

In [ ]:
# or we can use init subclass to generate some method (__repr__)

## 3 Descriptors

Learning about descriptors not only provides access to a larger toolset, it creates a deeper understanding of how Python works and an appreciation for the elegance of its design.
-- Raymond Hettinger, Python core developer and guru

In [ ]:
class Descriptor:
  def __get__(self, instance, owner):
    pass

  def __set__(self, instance, value):
    pass

  def __delete__(self, instance):
    pass


class Owner:
  # descriptor has to be defined as class attribute
  attr = Descriptor()

  def __init__(self, value):
    self.attr = value


instance = Owner(10)

In [ ]:
# Example 2
class Sensor:
  def __init__(self, name):
    self.storage_name = name

  def __get__(self, instance, owner):
    print(f'using __get__ {self, instance, owner}')
    return instance.__dict__[self.storage_name] + 5


class Termostat:
  obyvak = Sensor('obyvak')

  def __init__(self, obyvak):
    self.obyvak = obyvak

In [ ]:
t = Termostat(15)

In [ ]:
t.obyvak

In [ ]:
Termostat.obyvak.__get__(t, Termostat)

#### data vs nondata descriptor / overriding vs non-overriding

### Python equivalent for a dotted lookup in object.\_\_getattribute\_\_
#### https://docs.python.org/3/howto/descriptor.html#invocation-from-an-instance

In [ ]:
def find_name_in_mro(cls, name, default):
    "Emulate _PyType_Lookup() in Objects/typeobject.c"
    for base in cls.__mro__:
        if name in vars(base):
            return vars(base)[name]
    return default

def object_getattribute(obj, name):
    "Emulate PyObject_GenericGetAttr() in Objects/object.c"
    null = object()
    objtype = type(obj)
    cls_var = find_name_in_mro(objtype, name, null)
    descr_get = getattr(type(cls_var), '__get__', null)
    if descr_get is not null:
        if (hasattr(type(cls_var), '__set__')
            or hasattr(type(cls_var), '__delete__')):
            return descr_get(cls_var, obj, objtype)     # data descriptor
    if hasattr(obj, '__dict__') and name in vars(obj):
        return vars(obj)[name]                          # instance variable
    if descr_get is not null:
        return descr_get(cls_var, obj, objtype)         # non-data descriptor
    if cls_var is not null:
        return cls_var                                  # class variable
    raise AttributeError(name)


# The logic for a dotted lookup such as A.x is in type.__getattribute__().
# The steps are similar to those for object.__getattribute__() but the instance dictionary
# lookup is replaced by a search through the classâ€™s method resolution order.
# If a descriptor is found, it is invoked with desc.__get__(None, A).

### overriding descriptor: defining \_\_set\_\_

In [ ]:
# Example 3
class Sensor:
  def __init__(self, name):
    self.storage_name = name

  def __get__(self, instance, owner):
    print(f'using __get__ {self, instance, owner}')
    if instance is None: # fixes the class lookup
      return self
    return instance.__dict__[self.storage_name] + 5

  def __set__(self, instance, value):
    print(f'using __set__ {self, instance, value}')
    instance.__dict__[self.storage_name] = value

In [ ]:
class Termostat:
  obyvak = Sensor('obyvak')

  def __init__(self, obyvak):
    self.obyvak = obyvak

In [ ]:
t = Termostat(15)

In [ ]:
t.obyvak

In [ ]:
Termostat.obyvak.__get__(t, Termostat)

### \_\_set_name\_\_

In [ ]:
# Example 4
class Sensor:
  def __set_name__(self, owner, name):
    self.storage_name = name

  def __set__(self, instance, value):
    if value < 15:
      print(f'V {self.storage_name} je zima!')
    instance.__dict__[self.storage_name] = value

In [ ]:
class Termostat:
  obyvak = Sensor()
  koupelna = Sensor()
  loznice = Sensor()

  def __init__(self, obyvak, koupelna, loznice):
    self.obyvak = obyvak
    self.koupelna = koupelna
    self.loznice = loznice

In [ ]:
t = Termostat(15,15,15)

In [ ]:
t.obyvak = 12

### \_\_set_name\_\_ -> \_\_init_subclass\_\_

In [ ]:
# Example 6
class Sensor:
  def __set_name__(self, owner, name):
    print('__set_name__', name)

class Base:
  def __init_subclass__(cls):
    print('__init_subclass__', cls.__name__)

class Termostat(Base):
  obyvak = Sensor()

### Property is descriptor

In [ ]:
# Example 7
class T:
  def __init__(self, value):
    self._amount = value

  @property
  def amount(self):
    return self._amount

  @amount.setter
  def amount(self, value):
    self._amount = value

In [ ]:
t = T(10)

In [ ]:
t.amount

In [ ]:
T.amount.__set__(t, 11)

In [ ]:
T.amount.__get__(t, T)

### Python equivalent of property
### https://docs.python.org/3/howto/descriptor.html#invocation-from-an-instance

In [ ]:
class Property:
    "Emulate PyProperty_Type() in Objects/descrobject.c"

    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        self.fget = fget
        self.fset = fset
        self.fdel = fdel
        if doc is None and fget is not None:
            doc = fget.__doc__
        self.__doc__ = doc

    def __set_name__(self, owner, name):
        self.__name__ = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        if self.fget is None:
            raise AttributeError
        return self.fget(obj)

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError
        self.fdel(obj)

    def getter(self, fget):
        return type(self)(fget, self.fset, self.fdel, self.__doc__)

    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__)

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__)

In [ ]:
# Example 8
class T:
  def __init__(self, r):
    self.r = r

  @property
  def obvod(self):
    return 2 * 3.14 * self.r

In [ ]:
t = T(3)

In [ ]:
t.obvod = 10

### user-defined functions are descriptors

In [ ]:
# Example 9
class T:
  def __init__(self, r):
    self.r = r

  def obsah(self):
    return 3.14 * self.r ** 2

In [ ]:
t = T(10)

In [ ]:
t.obsah, T.obsah

In [ ]:
t.obsah.__func__ is T.obsah

In [ ]:
f = vars(T)['obsah']
f

In [ ]:
f.__get__(t, T)

#### functions are non-data/non-overriding descriptors

In [ ]:
hasattr(type(f), '__get__'), hasattr(type(f), '__set__')

In [ ]:
t.obsah = 'overwritten'
t.obsah

In [ ]:
del t.obsah

In [ ]:
# I'll be back
t.obsah

#### function stored on instance stays a function -is not bound

In [ ]:
t.g = lambda self: self
t.g

In [ ]:
# TypeError
t.g()

### functools.cached_property is descriptor

In [ ]:
from functools import cached_property